In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [5]:
from datasets import load_dataset, Dataset, Audio

az = load_dataset("google/fleurs", "az_az", split="train")
en  = load_dataset("google/fleurs", "en_us", split="train")

import pandas as pd
az_df = az.to_pandas()[["id", "audio", "transcription"]]
en_df  = en.to_pandas()[["id", "transcription"]].rename(columns={"transcription": "en_text"})

parallel = az_df.merge(en_df, on="id")
train_dataset = Dataset.from_pandas(parallel)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(train_dataset)
print(train_dataset[0]["en_text"])

Dataset({
    features: ['id', 'audio', 'transcription', 'en_text'],
    num_rows: 4632
})
news spread in the red lake community today as funerals for jeff weise and three of the nine victims were held that another student was arrested in connection with the school shootings of march 21


In [6]:
az_val = load_dataset("google/fleurs", "az_az", split="validation")
en_val  = load_dataset("google/fleurs", "en_us", split="validation")

az_val_df = az_val.to_pandas()[["id", "audio", "transcription"]]
en_val_df  = en_val.to_pandas()[["id", "transcription"]].rename(columns={"transcription": "en_text"})

parallel_val = az_val_df.merge(en_val_df, on="id")
val_dataset = Dataset.from_pandas(parallel_val)
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16000))
print(val_dataset)
print(val_dataset[0]["en_text"])

Dataset({
    features: ['id', 'audio', 'transcription', 'en_text'],
    num_rows: 1057
})
local authorities are warning residents in the vicinity of the plant to stay indoors turn off air-conditioners and not to drink tap water


In [7]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer

MODEL_NAME = "openai/whisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME,
    language="azerbaijani",
    task="translate"
)
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="chinese",
    task="translate"
)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

In [8]:
def preprocess(batch):

    audio_arrays = [x["array"] for x in batch["audio"]]
    batch["input_features"] = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors="np"
    ).input_features


    batch["labels"] = tokenizer(batch["en_text"]).input_ids

    return batch

In [9]:
train_dataset = train_dataset.map(
    preprocess, batched=True, batch_size=8,
    remove_columns=train_dataset.column_names
)
val_dataset = val_dataset.map(
    preprocess, batched=True, batch_size=8,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/4632 [00:00<?, ? examples/s]

Map:   0%|          | 0/1057 [00:00<?, ? examples/s]

In [10]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="azerbaijani",
    task="translate"
)

model.config.forced_decoder_ids = forced_decoder_ids
model.generation_config.forced_decoder_ids = forced_decoder_ids

model.config.suppress_tokens = []
model.generation_config.suppress_tokens = []

model.generation_config.language = "azerbaijani"
model.generation_config.task = "translate"

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [11]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

In [12]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [13]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00


In [15]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/whisper-az-en",

    max_steps=1000,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=4,

    logging_steps=25,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=1e-5,
    warmup_steps=50,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=True,
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
)

In [17]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    bleu = bleu_metric.compute(
        predictions=pred_str,
        references=[[reference] for reference in label_str]
    )

    return {
        "bleu": bleu["score"]
    }

In [23]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

In [24]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Bleu
250,4.226550,2.785830,4.871792


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]